# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kishan992/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Binary Classification with Probability Scoring

**Which one and why:**
We frame this project as a **Binary Classification** task paired with **Probability Scoring**:

* **Supervised Learning:** We leverage historical search performance logs (clicks, impressions, average positions, CTR) up to our decision cutoff date (`2026-06-25`).
* **Classification (1 vs 0):** The model assigns a binary label to each web page:
  * **1 = Decaying Target** (Content traffic collapses post-cutoff below 50% of baseline volume).
  * **0 = Stable / Growing** (Content maintains performance; no immediate refresh required).
* **Probability Scoring:** Editorial teams cannot rewrite thousands of pages simultaneously. LightGBM outputs a continuous probability score ($P(\text{declining}) \in [0.0, 1.0]$) to order content into a prioritized action queue.

In [1]:
# Section 1 Code: Define ML Task Parameters
task_type = "Binary Classification with Probability Scoring"
model_goal = "Predict web page traffic decay risk (0.0 to 1.0 probability)"
decision_cutoff = "2026-06-25"

print(f"✓ Task Type       : {task_type}")
print(f"✓ Model Goal      : {model_goal}")
print(f"✓ Decision Cutoff : {decision_cutoff}")

✓ Task Type       : Binary Classification with Probability Scoring
✓ Model Goal      : Predict web page traffic decay risk (0.0 to 1.0 probability)
✓ Decision Cutoff : 2026-06-25


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**What we are predicting:**
We predict a binary target column `is_declining_target` (1 or 0), indicating whether a content item suffers a severe post-cutoff click collapse.

**Where the label comes from:**
The label is constructed by comparing pre-cutoff historical clicks ($t \le \text{2026-06-25}$) against post-cutoff clicks ($t > \text{2026-06-25}$):

* **Label = 1 (Decaying):** $\text{post\_clicks} < 0.5 \times \text{pre\_clicks}$ (Traffic drops by more than 50%).
* **Label = 0 (Stable/Growth):** $\text{post\_clicks} \ge 0.5 \times \text{pre\_clicks}$ (Traffic remains healthy).

In [2]:
import pandas as pd

# Demonstrating the Target Labeling Proxy Rule
demo_data = pd.DataFrame({
    'content_hash_id': ['content_a', 'content_b', 'content_c'],
    'pre_clicks': [400, 280, 150],
    'post_clicks': [180, 290, 60]
})

# Apply exact ground truth target logic
demo_data['is_declining_target'] = (demo_data['post_clicks'] < (demo_data['pre_clicks'] * 0.5)).astype(int)

print("Target Proxy Rule Demonstration:")
print(demo_data.to_string(index=False))

Target Proxy Rule Demonstration:
content_hash_id  pre_clicks  post_clicks  is_declining_target
      content_a         400          180                    1
      content_b         280          290                    0
      content_c         150           60                    1


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Ranking Metric:** Precision@K (Top 10%, Top 20%, Top 30%) & ROC-AUC

**What number means 'good'?**
* **Target:** **> 75% Precision@Top 10%** (When recommending the top 10% highest-risk pages for refresh, at least 3 out of 4 must be true declining assets).
* **Secondary Targets:** ROC-AUC $> 0.80$, Log-Loss $< 0.50$, and Spearman Rank Correlation ($\rho$) $> +0.50$.

**Why this metric:**
Editorial teams have fixed weekly bandwidth. Precision@K measures the exact accuracy of the top recommendations passed to writers, ensuring zero wasted rewrite budget on healthy pages.

In [3]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score

# Simulating evaluation queue performance
np.random.seed(42)
y_true = np.random.choice([0, 1], size=100, p=[0.53, 0.47])  # 47% decay rate matching warehouse
random_scores = np.random.uniform(0, 1, size=100)

eval_df = pd.DataFrame({'is_declining_target': y_true, 'predicted_score': random_scores})
top_10 = eval_df.sort_values(by='predicted_score', ascending=False).head(10)
p10 = top_10['is_declining_target'].mean() * 100.0

print("=" * 60)
print("SECTION 3 METRIC EVALUATION DEMONSTRATION")
print("=" * 60)
print(f"• Sample Monitored Queue Size    : {len(eval_df)}")
print(f"• Random Baseline Precision@10%  : {p10:.2f}%")
print(f"• ML Model Precision@10% Target  : > 75.0%")
print("=" * 60)

SECTION 3 METRIC EVALUATION DEMONSTRATION
• Sample Monitored Queue Size    : 100
• Random Baseline Precision@10%  : 50.00%
• ML Model Precision@10% Target  : > 75.0%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**What is one row in this DataFrame?**
One row represents **one unique content entity (`content_hash_id`)** aggregated over the pre-cutoff observation window ($t \le \text{2026-06-25}$).

**Why this unit?**
Content refreshes (rewriting titles, updating outdated statistics, expanding depth) are executed at the individual URL/content item level.

In [4]:
import duckdb
import pandas as pd
import os
import glob
from huggingface_hub import snapshot_download

# Authenticate HF Token
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")

local_dir = snapshot_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset", token=HF_TOKEN)
all_parquet = glob.glob(os.path.join(local_dir, "**", "*.parquet"), recursive=True)
parquet_files = [f for f in all_parquet if "fact_content_daily_performance" in f]

con = duckdb.connect(database=':memory:')

# Query showing exact unit of analysis
df_grain_demo = con.execute(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_clicks) AS pre_clicks,
        SUM(gsc_impressions) AS pre_impressions,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS pre_avg_position
    FROM read_parquet({parquet_files}, union_by_name=True)
    WHERE report_date <= '2026-06-25'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
    LIMIT 5
""").df()

print("Unit of Analysis Verification (1 Row = 1 content_hash_id):")
print(df_grain_demo.to_string(index=False))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unit of Analysis Verification (1 Row = 1 content_hash_id):
         content_hash_id          client_hash_id  pre_clicks  pre_impressions  pre_avg_position
content_08bfdd6c0a86993f client_f623b01661d4bfe4         4.0            644.0         14.447197
content_a23309f704c98e59 client_f623b01661d4bfe4        38.0            770.0          5.378830
content_0ef92dd8ea773a64 client_f623b01661d4bfe4         3.0           3566.0         76.721188
content_896651a179d0c7f8 client_f623b01661d4bfe4        58.0           3242.0         20.305316
content_0f97f04ce6927790 client_f623b01661d4bfe4         6.0           1750.0         71.009185


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why simple `if/else` rules fail:**
* **Volume Over-Prioritization:** Fixed rules prioritize pages with raw high impression counts, incorrectly flagging healthy high-traffic pages.
* **Non-Linear SERP Interactions:** Search rank position decay (e.g., dropping from rank 3 to rank 8) exhibits non-linear CTR drops that fixed thresholds cannot capture.
* **Grouped Generalization:** Rules cannot adapt across different client domain authorities, whereas LightGBM generalizes patterns out-of-fold across client groups (`GroupKFold`).

In [5]:
# Demonstrating rule failure vs ML multidimensional signal
edge_cases = pd.DataFrame({
    'content_hash_id': ['page_high_vol_healthy', 'page_rank_decay'],
    'pre_clicks': [5000, 300],
    'pre_impressions': [100000, 8000],
    'pre_avg_position': [2.1, 8.5],
    'post_clicks': [4800, 80] # Page 2 collapsed!
})

# Fixed rule logic: "Flag if pre_impressions > 50,000"
edge_cases['rule_flag'] = (edge_cases['pre_impressions'] > 50000).astype(int)
# True target logic
edge_cases['is_declining_target'] = (edge_cases['post_clicks'] < (edge_cases['pre_clicks'] * 0.5)).astype(int)

print("Rule Failure Analysis:")
print(edge_cases[['content_hash_id', 'rule_flag', 'is_declining_target']].to_string(index=False))

Rule Failure Analysis:
      content_hash_id  rule_flag  is_declining_target
page_high_vol_healthy          1                    0
      page_rank_decay          0                    1


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.